# Cleaning — Egyptian Real Estate Dataset

**Objective:** Group similar Egyptian properties using the K-Means clustering algorithm.  
The output will be the cleaned dataset enriched with a new `cluster` column, saved for use in the Recommendation System stage.

**Input:** `../data/processed/cleaned_all_egypt.csv`  
**Output:** `../data/processed/clustered_all_egypt.csv`

In [1]:
# Load Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load dataset
dataset_path = "../../data/raw/properties/all_egypt.csv"

df = pd.read_csv(dataset_path)

# Display first 5 rows
df.head()

C:\Users\HELAL\AppData\Local\Temp\ipykernel_13976\3856943173.py:4: DtypeWarning: Columns (0: id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(dataset_path)


,id,title,price,currency,price_period,price_type,price_raw_value,property_type,location,latitude,...,bedrooms,bathrooms,size,amenities,category,page,scraped_at_utc,record_type,min_price,max_price
0,52974566,Luxurious Living in the Heart of Hyde Park!,50000.0,EGP,monthly,rent_price,50000.0,Apartment,"Hyde Park, New Cairo City, Cairo",29.995922,...,3.0,3.0,189.0,"['BA', 'SE', 'WC', 'LB']",rent,1,2026-02-19T16:31:09.727088+00:00,property,NaN,NaN
1,8653638,Apartment for Rent in Al Marasem Compound,35000.0,EGP,monthly,rent_price,35000.0,Apartment,"Moon Residences, Fifth Square, The 5th Settlem...",30.051462,...,3.0,2.0,170.0,"['BA', 'BW', 'AC', 'BK', 'CP']",rent,1,2026-02-19T16:31:09.727088+00:00,property,NaN,NaN
2,52974180,Flat Ultra Super Lux First Use For Rent-Villette,70000.0,EGP,monthly,rent_price,70000.0,Apartment,"Villette, 5th Settlement Compounds, The 5th Se...",30.022596,...,3.0,3.0,161.0,"['BA', 'AC', 'CP', 'BK', 'SE', 'WC', 'BL', 'LB']",rent,1,2026-02-19T16:31:09.727088+00:00,property,NaN,NaN
3,52969227,Studio for Rent — South Lotus (New Cairo),22000.0,EGP,monthly,rent_price,22000.0,Apartment,"South Lotus, El Lotus, New Cairo City, Cairo",30.019306,...,1.0,1.0,80.0,"['BA', 'BW', 'AC', 'CP', 'BK', 'SP', 'SE', 'SY...",rent,1,2026-02-19T16:31:09.727088+00:00,property,NaN,NaN
4,52969105,Studio for Rent — Hyde Park New Cairo First Us,35000.0,EGP,monthly,rent_price,35000.0,Apartment,"Hyde Park, New Cairo City, Cairo",29.995922,...,1.0,1.0,80.0,"['BA', 'BW', 'AC', 'CP', 'BK', 'CO', 'LB', 'BL...",rent,1,2026-02-19T16:31:09.727088+00:00,property,NaN,NaN


In [3]:
# Display last 5 rows
df.tail()

,id,title,price,currency,price_period,price_type,price_raw_value,property_type,location,latitude,...,bedrooms,bathrooms,size,amenities,category,page,scraped_at_utc,record_type,min_price,max_price
64101,3cd8322e-e019-438f-a06d-67018fcb4a79,Sarayat El Katameya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.977171,...,NaN,NaN,NaN,NaN,new_projects,47,2026-02-19T18:23:50.493989+00:00,project,NaN,NaN
64102,92f38ba5-3a81-4523-b1f2-f66f24581338,Vee Muruj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.209261,...,NaN,NaN,NaN,NaN,new_projects,47,2026-02-19T18:23:50.493989+00:00,project,NaN,NaN
64103,a2684514-84c5-4e99-a342-8033a975ab83,Valore Sheraton,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.105271,...,NaN,NaN,NaN,NaN,new_projects,47,2026-02-19T18:23:50.493989+00:00,project,NaN,NaN
64104,9cff880a-1ece-4cc2-9208-a0cdbfe23a49,Arena City Towers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.054694,...,NaN,NaN,NaN,NaN,new_projects,47,2026-02-19T18:23:50.493989+00:00,project,NaN,NaN
64105,fbcf2fb6-7b47-4eec-8d43-7d0798282e20,M11 Meraki Residence,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,...,NaN,NaN,NaN,NaN,new_projects,47,2026-02-19T18:23:50.493989+00:00,project,NaN,NaN


In [4]:
# Check data types, missing values, and memory usage
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64106 entries, 0 to 64105
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               64106 non-null  object 
 1   title            64106 non-null  str    
 2   price            62977 non-null  float64
 3   currency         62985 non-null  str    
 4   price_period     62985 non-null  str    
 5   price_type       62985 non-null  str    
 6   price_raw_value  62985 non-null  float64
 7   property_type    62985 non-null  str    
 8   location         62985 non-null  str    
 9   latitude         64106 non-null  float64
 10  longitude        64106 non-null  float64
 11  images           64106 non-null  str    
 12  share_url        62985 non-null  str    
 13  bedrooms         52593 non-null  float64
 14  bathrooms        60228 non-null  float64
 15  size             62985 non-null  float64
 16  amenities        62985 non-null  str    
 17  category         64106 

In [5]:
# Summary statistics of numerical features
df.describe()

,price,price_raw_value,latitude,longitude,bedrooms,bathrooms,size,page,min_price,max_price
count,6.297700e+04,6.298500e+04,64106.000000,64106.000000,52593.000000,60228.000000,6.298500e+04,64106.000000,0.0,0.0
mean,1.013712e+07,1.013585e+07,29.936054,31.291449,3.010686,2.599638,4.621353e+02,337.002933,NaN,NaN
std,4.106258e+07,4.106012e+07,1.483013,1.167402,1.343315,1.388999,1.460059e+04,221.131582,NaN,NaN
min,1.000000e+03,1.000000e+03,0.000000,0.000000,1.000000,1.000000,1.000000e+00,1.000000,NaN,NaN
25%,7.000000e+04,7.000000e+04,29.999271,31.028166,2.000000,2.000000,1.060000e+02,150.000000,NaN,NaN
50%,4.850000e+05,4.840000e+05,30.024694,31.449591,3.000000,2.000000,1.670000e+02,310.000000,NaN,NaN
75%,9.050000e+06,9.048530e+06,30.061470,31.538992,4.000000,3.000000,2.570000e+02,508.000000,NaN,NaN
max,1.000000e+09,1.000000e+09,31.571518,34.632736,7.000000,7.000000,2.520000e+06,800.000000,NaN,NaN


---
## 1. Remove Unnecessary Columns

In [6]:
columns_to_remove = [
    "min_price",
    "max_price",
    "page",
    "scraped_at_utc",
    "record_type"
]

df = df.drop(columns=columns_to_remove)

print("Remaining Columns:", len(df.columns))
display(df.head())

Remaining Columns: 18


,id,title,price,currency,price_period,price_type,price_raw_value,property_type,location,latitude,longitude,images,share_url,bedrooms,bathrooms,size,amenities,category
0,52974566,Luxurious Living in the Heart of Hyde Park!,50000.0,EGP,monthly,rent_price,50000.0,Apartment,"Hyde Park, New Cairo City, Cairo",29.995922,31.554127,['https://static.shared.propertyfinder.eg/medi...,https://www.propertyfinder.eg/en/plp/rent/apar...,3.0,3.0,189.0,"['BA', 'SE', 'WC', 'LB']",rent
1,8653638,Apartment for Rent in Al Marasem Compound,35000.0,EGP,monthly,rent_price,35000.0,Apartment,"Moon Residences, Fifth Square, The 5th Settlem...",30.051462,31.538992,['https://static.shared.propertyfinder.eg/medi...,https://www.propertyfinder.eg/en/plp/rent/apar...,3.0,2.0,170.0,"['BA', 'BW', 'AC', 'BK', 'CP']",rent
2,52974180,Flat Ultra Super Lux First Use For Rent-Villette,70000.0,EGP,monthly,rent_price,70000.0,Apartment,"Villette, 5th Settlement Compounds, The 5th Se...",30.022596,31.545815,['https://static.shared.propertyfinder.eg/medi...,https://www.propertyfinder.eg/en/plp/rent/apar...,3.0,3.0,161.0,"['BA', 'AC', 'CP', 'BK', 'SE', 'WC', 'BL', 'LB']",rent
3,52969227,Studio for Rent — South Lotus (New Cairo),22000.0,EGP,monthly,rent_price,22000.0,Apartment,"South Lotus, El Lotus, New Cairo City, Cairo",30.019306,31.513653,['https://static.shared.propertyfinder.eg/medi...,https://www.propertyfinder.eg/en/plp/rent/apar...,1.0,1.0,80.0,"['BA', 'BW', 'AC', 'CP', 'BK', 'SP', 'SE', 'SY...",rent
4,52969105,Studio for Rent — Hyde Park New Cairo First Us,35000.0,EGP,monthly,rent_price,35000.0,Apartment,"Hyde Park, New Cairo City, Cairo",29.995922,31.554127,['https://static.shared.propertyfinder.eg/medi...,https://www.propertyfinder.eg/en/plp/rent/apar...,1.0,1.0,80.0,"['BA', 'BW', 'AC', 'CP', 'BK', 'CO', 'LB', 'BL...",rent


---
## 2. Remove Records with Missing Values in Essential Features

In [7]:
required_columns = [
    "price",
    "price_period",
    "currency",
    "amenities",
    "size",
    "price_raw_value",
    "price_type",
    "share_url",
    "location",
    "property_type"
]

before_rows = len(df)

df = df.dropna(subset=required_columns)

after_rows = len(df)

print(f"Rows Before Cleaning : {before_rows}")
print(f"Rows After Cleaning  : {after_rows}")
print(f"Removed Rows         : {before_rows-after_rows}")

Rows Before Cleaning : 64106
Rows After Cleaning  : 62977
Removed Rows         : 1129


---
## 3.Preserve Missing Values in Bedrooms and Bathrooms

In [8]:
print("Bedrooms Missing :", df["bedrooms"].isna().sum())
print("Bathrooms Missing:", df["bathrooms"].isna().sum())

print("\nNo imputation will be performed.")

Bedrooms Missing : 10389
Bathrooms Missing: 2756

No imputation will be performed.


---
## 4. Verify Missing Values

In [8]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": round(df.isnull().mean()*100,2)
})

missing = missing[missing["Missing Values"] > 0]

display(missing.sort_values("Percentage", ascending=False))

,Missing Values,Percentage
bedrooms,10389,16.50
bathrooms,2756,4.38


---
## 5. Verify Dataset Information

In [9]:
print("Dataset Shape")

display(df.shape)

print("\nDataset Information")

df.info()

Dataset Shape


(62977, 18)


Dataset Information
<class 'pandas.DataFrame'>
Index: 62977 entries, 0 to 62984
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               62977 non-null  object 
 1   title            62977 non-null  str    
 2   price            62977 non-null  float64
 3   currency         62977 non-null  str    
 4   price_period     62977 non-null  str    
 5   price_type       62977 non-null  str    
 6   price_raw_value  62977 non-null  float64
 7   property_type    62977 non-null  str    
 8   location         62977 non-null  str    
 9   latitude         62977 non-null  float64
 10  longitude        62977 non-null  float64
 11  images           62977 non-null  str    
 12  share_url        62977 non-null  str    
 13  bedrooms         52588 non-null  float64
 14  bathrooms        60221 non-null  float64
 15  size             62977 non-null  float64
 16  amenities        62977 non-null  str    
 17  categor

---
## 6. Display Final Dataset

In [10]:
display(df.head())

,id,title,price,currency,price_period,price_type,price_raw_value,property_type,location,latitude,longitude,images,share_url,bedrooms,bathrooms,size,amenities,category
0,52974566,Luxurious Living in the Heart of Hyde Park!,50000.0,EGP,monthly,rent_price,50000.0,Apartment,"Hyde Park, New Cairo City, Cairo",29.995922,31.554127,['https://static.shared.propertyfinder.eg/medi...,https://www.propertyfinder.eg/en/plp/rent/apar...,3.0,3.0,189.0,"['BA', 'SE', 'WC', 'LB']",rent
1,8653638,Apartment for Rent in Al Marasem Compound,35000.0,EGP,monthly,rent_price,35000.0,Apartment,"Moon Residences, Fifth Square, The 5th Settlem...",30.051462,31.538992,['https://static.shared.propertyfinder.eg/medi...,https://www.propertyfinder.eg/en/plp/rent/apar...,3.0,2.0,170.0,"['BA', 'BW', 'AC', 'BK', 'CP']",rent
2,52974180,Flat Ultra Super Lux First Use For Rent-Villette,70000.0,EGP,monthly,rent_price,70000.0,Apartment,"Villette, 5th Settlement Compounds, The 5th Se...",30.022596,31.545815,['https://static.shared.propertyfinder.eg/medi...,https://www.propertyfinder.eg/en/plp/rent/apar...,3.0,3.0,161.0,"['BA', 'AC', 'CP', 'BK', 'SE', 'WC', 'BL', 'LB']",rent
3,52969227,Studio for Rent — South Lotus (New Cairo),22000.0,EGP,monthly,rent_price,22000.0,Apartment,"South Lotus, El Lotus, New Cairo City, Cairo",30.019306,31.513653,['https://static.shared.propertyfinder.eg/medi...,https://www.propertyfinder.eg/en/plp/rent/apar...,1.0,1.0,80.0,"['BA', 'BW', 'AC', 'CP', 'BK', 'SP', 'SE', 'SY...",rent
4,52969105,Studio for Rent — Hyde Park New Cairo First Us,35000.0,EGP,monthly,rent_price,35000.0,Apartment,"Hyde Park, New Cairo City, Cairo",29.995922,31.554127,['https://static.shared.propertyfinder.eg/medi...,https://www.propertyfinder.eg/en/plp/rent/apar...,1.0,1.0,80.0,"['BA', 'BW', 'AC', 'CP', 'BK', 'CO', 'LB', 'BL...",rent


---
## 7. Save Clean Dataset

In [12]:
output_path = "../../data/processed/cleaned_all_egypt.csv"

df.to_csv(output_path, index=False)

print("Clean dataset saved successfully.")
print(output_path)

Clean dataset saved successfully.
../../data/processed/cleaned_all_egypt.csv


In [11]:
# Display all unique values in the 'category' column
print(df["category"].unique())

<StringArray>
['rent', 'buy', 'commercial_rent', 'commercial_buy']
Length: 4, dtype: str
